In [1]:
import numpy as np
import scanpy as sc
from scipy import sparse
import anndata as ad

import os
import pandas as pd

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
)
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

# Nếu bạn đã cài lightgbm, xgboost:
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

In [3]:


adata1 = sc.read_h5ad('D:/VietBao2025/Methods/Dataset/MOSTA/Chen-Stereo_seq-E15.5-s1.h5ad')
adata2 = sc.read_h5ad('D:/VietBao2025/Methods/Dataset/MOSTA/Chen-Stereo_seq-E15.5-s2.h5ad')
# adata1 = sc.read_h5ad('D:/VietBao2025/Methods/Dataset/CANCER/A73rectum_1_filter.h5ad')
# adata2 = sc.read_h5ad('D:/VietBao2025/Methods/Dataset/CANCER/A73rectum_2_filter.h5ad')
# adata1 = sc.read_h5ad('D:/VietBao2025/Methods/Dataset/Maynard/151507.h5ad')
# adata2 = sc.read_h5ad('D:/VietBao2025/Methods/Dataset/Maynard/151508.h5ad')
# adata1 = sc.read_h5ad('D:/VietBao2025/Methods/Dataset/CANCER/S69_filter.h5ad') #Arutyunyan
# adata2 = sc.read_h5ad('D:/VietBao2025/Methods/Dataset/CANCER/S70_filter.h5ad')
adata_train = adata1
adata_test = adata2

In [4]:
adata_train

AnnData object with n_obs × n_vars = 6000 × 28798
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'annotation', 'Regulon - AI987944', 'Brain_shapes', 'Face_shapes', 'Heart_shapes', 'Lung_shapes', 'Liver_shapes', 'Shapes_shapes', 'Belly_shapes', 'Back_shapes', 'region'
    var: 'n_cells', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts'
    uns: 'region_colors'
    obsm: 'spatial', 'spatial_back'
    varm: 'PCs'

In [6]:
print(adata_train.X[:10])

  (0, 13)	1.0
  (0, 17)	1.0
  (0, 20)	12.0
  (0, 21)	7.0
  (0, 26)	1.0
  (0, 28)	1.0
  (0, 30)	1.0
  (0, 32)	1.0
  (0, 34)	2.0
  (0, 36)	4.0
  (0, 37)	1.0
  (0, 39)	2.0
  (0, 47)	1.0
  (0, 50)	1.0
  (0, 54)	1.0
  (0, 57)	2.0
  (0, 61)	1.0
  (0, 64)	1.0
  (0, 65)	1.0
  (0, 66)	2.0
  (0, 67)	2.0
  (0, 68)	1.0
  (0, 70)	2.0
  (0, 71)	1.0
  (0, 75)	1.0
  :	:
  (9, 15558)	1.0
  (9, 15567)	2.0
  (9, 15760)	2.0
  (9, 15862)	1.0
  (9, 16018)	1.0
  (9, 16019)	4.0
  (9, 16327)	1.0
  (9, 16387)	1.0
  (9, 16443)	1.0
  (9, 16480)	1.0
  (9, 16663)	1.0
  (9, 16720)	1.0
  (9, 17204)	1.0
  (9, 17365)	1.0
  (9, 18584)	2.0
  (9, 18868)	1.0
  (9, 18909)	1.0
  (9, 18998)	1.0
  (9, 19051)	1.0
  (9, 19834)	3.0
  (9, 20111)	1.0
  (9, 20551)	1.0
  (9, 21155)	1.0
  (9, 22122)	1.0
  (9, 24991)	1.0


In [7]:
adata_train.obs

,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,annotation,Regulon - AI987944,Brain_shapes,Face_shapes,Heart_shapes,Lung_shapes,Liver_shapes,Shapes_shapes,Belly_shapes,Back_shapes,region
cell_name,,,,,,,,,,,,,,,
227_207,1751,7.468513271496337,3933,8.277411998949004,Muscle,0.34274798537686,False,True,False,False,False,False,False,False,Face_shapes
231_158,1124,7.025538314638521,2401,7.784057002639929,Connective tissue,0.0570454140334814,False,True,False,False,False,False,False,False,Face_shapes
341_183,902,6.805722553416985,1330,7.193685818395112,Liver,0.1608166609379092,False,False,False,False,True,False,False,False,Liver_shapes
164_71,1305,7.174724309836376,2931,7.983440063006542,Meninges,0.3415884500875779,False,False,False,False,False,False,False,True,Back_shapes
435_106,1242,7.1252830915107115,2490,7.820439515262181,Cartilage,0.2402527298617984,False,False,False,False,False,False,False,True,Back_shapes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
405_277,2730,7.912423121473705,6556,8.788288460263619,Epidermis,0.460555915721495,False,False,False,False,False,False,False,True,Back_shapes
403_26,850,6.7464121285733745,1300,7.170888478512505,Epidermis,0.4381954153124103,False,False,False,False,False,False,False,True,Back_shapes
191_188,1590,7.372118028337787,3901,8.269244521183056,Muscle,0.6471578801234152,False,True,False,False,False,False,False,False,Face_shapes


In [8]:
adata_train

AnnData object with n_obs × n_vars = 6000 × 28798
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'annotation', 'Regulon - AI987944', 'Brain_shapes', 'Face_shapes', 'Heart_shapes', 'Lung_shapes', 'Liver_shapes', 'Shapes_shapes', 'Belly_shapes', 'Back_shapes', 'region'
    var: 'n_cells', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts'
    uns: 'region_colors'
    obsm: 'spatial', 'spatial_back'
    varm: 'PCs'

In [9]:
adata_test

AnnData object with n_obs × n_vars = 5000 × 28154
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'annotation', 'Regulon - 1700080O16Rik', 'Brain_shapes', 'Face_shapes', 'Shapes_shapes', 'Lung_shapes', 'Belly_shapes', 'Liver_shapes', 'Back_shapes', 'region', 'Heart_shapes'
    var: 'n_cells', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts'
    uns: 'Back_shapes', 'Belly_shapes', 'Brain_shapes', 'Face_shapes', 'Face_shapes_colors', 'Heart_shapes', 'Liver_shapes', 'Lung_shapes', 'Shapes_shapes', 'region_colors'
    obsm: 'spatial', 'spatial_back'
    varm: 'PCs'

In [10]:
adata_train.X

<6000x28798 sparse matrix of type '<class 'numpy.float32'>'
	with 10658998 stored elements in Compressed Sparse Row format>

In [12]:

# ============================================================
# 1. HVG + X matrix (giữ nguyên logic của bạn)
# ============================================================

def select_hvg_from_train(adata_train, n_top_genes=5000):
    """
    Chọn n_top_genes gene biến thiên cao (HVG) từ tập train.
    Dùng flavor='cell_ranger' vì dữ liệu đã log/normalize upstream.
    """
    
    # sc.pp.normalize_total(adata_train, target_sum=1e4)
    # sc.pp.log1p(adata_train)
    sc.pp.highly_variable_genes(
        adata_train,
        n_top_genes=n_top_genes,
        flavor="cell_ranger",#với tập mosta
        # flavor="seurat_v3",
        subset=True, 
        inplace=True
    )
    hvg_genes = adata_train.var[adata_train.var["highly_variable"]].index
    hvg_genes = list(hvg_genes[:n_top_genes])
    return hvg_genes


def build_X_from_genes(adata, selected_genes):
    """
    Tạo ma trận feature X có shape (n_cells, len(selected_genes))
    Nếu gene không tồn tại trong adata, cột tương ứng = 0.
    """
    n_cells = adata.n_obs
    gene2idx = {g: i for i, g in enumerate(adata.var_names)}
    is_sparse = sparse.issparse(adata.X)

    cols = []
    for g in selected_genes:
        if g in gene2idx:
            idx = gene2idx[g]
            col = adata.X[:, idx]
            if is_sparse:
                col = col.toarray()
            col = np.asarray(col).reshape(n_cells, 1)
        else:
            col = np.zeros((n_cells, 1), dtype=np.float32)
        cols.append(col)

    X = np.concatenate(cols, axis=1)
    return X.astype(np.float32)


# def build_label_mapping(adata_train, adata_test, label_key="niche"):
#     """
#     Tạo mapping label -> id chung cho cả train & test để tránh lệch mã.
#     """
#     all_labels = pd.concat(
#         [adata_train.obs[label_key].astype(str),
#          adata_test.obs[label_key].astype(str)]
#     )
#     categories = sorted(all_labels.unique())
#     label2id = {lab: i for i, lab in enumerate(categories)}
#     return label2id

def build_label_mapping(adata_train, label_key="region"):
    labels = adata_train.obs[label_key].astype(str)
    categories = sorted(labels.unique())
    label2id = {lab: i for i, lab in enumerate(categories)}
    return label2id



def get_y_from_adata(adata, label_key, label2id):
    labels = adata.obs[label_key].astype(str).map(label2id).values
    return labels.astype(int)


# ============================================================
# 2. Eval + lưu CSV
# ============================================================

def evaluate_and_save(model_name, y_true, y_pred, save_dir="Mosta_ML_labels"):
    """
    Lưu file CSV <model_name>_labels.csv và in các metric.
    """
    os.makedirs(save_dir, exist_ok=True)
    y_true = np.squeeze(y_true)
    y_pred = np.squeeze(y_pred)

    df = pd.DataFrame({
        "true_labels": y_true,
        "predicted_labels": y_pred,
    })
    csv_name = f"{model_name.lower()}_labels.csv"
    csv_path = os.path.join(save_dir, csv_name)
    df.to_csv(csv_path, index=False)
    print(f"Saved predictions to: {csv_path}")

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")
    pre = precision_score(y_true, y_pred, average="macro")
    rec = recall_score(y_true, y_pred, average="macro")
    ari = adjusted_rand_score(y_true, y_pred)
    nmi = normalized_mutual_info_score(y_true, y_pred)

    print(f"{model_name} metrics:")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  F1 (macro): {f1:.4f}")
    print(f"  Precision (macro): {pre:.4f}")
    print(f"  Recall (macro): {rec:.4f}")
    print(f"  ARI: {ari:.4f}")
    print(f"  NMI: {nmi:.4f}")
    print("-" * 50)

    return {
        "accuracy": acc,
        "f1_macro": f1,
        "precision_macro": pre,
        "recall_macro": rec,
        "ari": ari,
        "nmi": nmi,
    }


# ============================================================
# 3. Hàm train chung cho các model ML
# ============================================================

def train_and_eval_classifiers(
    adata_train,
    adata_test,
    label_key="region",
    n_top_genes=5000,
):
    # # 1) Normalize
    # sc.pp.normalize_total(adata_train, target_sum=1e4)

    # # # 2) Log transform
    # sc.pp.log1p(adata_train)

    # 1) HVG từ train
    hvg_genes = select_hvg_from_train(adata_train, n_top_genes=n_top_genes)

    # 2) X_train, X_test
    X_train = build_X_from_genes(adata_train, hvg_genes)

    # sc.pp.normalize_total(adata_test, target_sum=1e4)

    # sc.pp.log1p(adata_test)
    
    X_test  = build_X_from_genes(adata_test,  hvg_genes)

    # 3) y_train, y_test với mapping chung
    label2id = build_label_mapping(adata_train, label_key=label_key)
    y_train = get_y_from_adata(adata_train, label_key, label2id)
    y_test  = get_y_from_adata(adata_test,  label_key, label2id)

    n_classes = len(label2id)

    # 4) Chuẩn hoá feature cho các model tuyến tính / KNN
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    results = {}

    # --------- 1. KNN ----------
    knn = KNeighborsClassifier(n_neighbors=5, weights="distance")
    knn.fit(X_train_scaled, y_train)
    y_pred_knn = knn.predict(X_test_scaled)
    results["KNN"] = evaluate_and_save("KNN", y_test, y_pred_knn)

    # --------- 2. LightGBM ----------
    lgbm = LGBMClassifier(
        objective="multiclass",
        num_class=n_classes,
        n_estimators=300,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
    )
    lgbm.fit(X_train, y_train)
    y_pred_lgbm = lgbm.predict(X_test)
    results["LightGBM"] = evaluate_and_save("LightGBM", y_test, y_pred_lgbm)

    # --------- 3. XGBoost ----------
    xgb = XGBClassifier(
        objective="multi:softprob",
        num_class=n_classes,
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        eval_metric="mlogloss",
    )
    xgb.fit(X_train, y_train)

    # y_pred_xgb = xgb.predict(X_test)   # SAI – trả về ma trận (n, n_class)
    y_pred_xgb = xgb.predict_proba(X_test).argmax(axis=1)  # ĐÚNG

    results["XGBoost"] = evaluate_and_save("XGBoost", y_test, y_pred_xgb)

    # --------- 4. SGD (logistic regression kiểu online) ----------
    sgd = SGDClassifier(
        loss="log_loss",
        max_iter=2000,
        tol=1e-3,
        n_jobs=-1,
    )
    sgd.fit(X_train_scaled, y_train)
    y_pred_sgd = sgd.predict(X_test_scaled)
    results["SGD"] = evaluate_and_save("SGD", y_test, y_pred_sgd)

    # --------- 5. Decision Tree ----------
    dt = DecisionTreeClassifier(
        max_depth=None,
        min_samples_split=10,
        min_samples_leaf=5,
    )
    dt.fit(X_train, y_train)
    y_pred_dt = dt.predict(X_test)
    results["DecisionTree"] = evaluate_and_save("DecisionTree", y_test, y_pred_dt)

    # --------- 6. Naive Bayes ----------
    # Dữ liệu là continuous (log-expression) → dùng GaussianNB
    nb = GaussianNB()
    nb.fit(X_train, y_train)
    y_pred_nb = nb.predict(X_test)
    results["NaiveBayes"] = evaluate_and_save("NaiveBayes", y_test, y_pred_nb)

    return results


# ============================================================
# 4. Gọi pipeline với adata_train, adata_test
# ============================================================

# Giả sử bạn đã load:
# adata_train, adata_test

results_all = train_and_eval_classifiers(
    adata_train,
    adata_test,
    label_key="region",   # hoặc "annotation" nếu bạn muốn
    n_top_genes=5000,
)


Saved predictions to: Mosta_ML_labels\knn_labels.csv
KNN metrics:
  Accuracy: 0.2334
  F1 (macro): 0.1837
  Precision (macro): 0.2383
  Recall (macro): 0.2729
  ARI: 0.0123
  NMI: 0.0825
--------------------------------------------------
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.112902 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 29558
[LightGBM] [Info] Number of data points in the train set: 6000, number of used features: 3243
[LightGBM] [Info] Start training from score -1.214584
[LightGBM] [Info] Start training from score -2.603690
[LightGBM] [Info] Start training from score -1.303793
[LightGBM] [Info] Start training from score -1.578262
[LightGBM] [Info] Start training from score -2.841582
[LightGBM] [Info] Start training from score -2.743677
[LightGBM] [Info] Start training from score -3.605765
[LightGBM] [Info] Start training from score -6.502290
[LightGBM] [Warning] No further splits 

c:\Users\ADMIN\anaconda3\envs\tfm\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Saved predictions to: Mosta_ML_labels\xgboost_labels.csv
XGBoost metrics:
  Accuracy: 0.7172
  F1 (macro): 0.5523
  Precision (macro): 0.6702
  Recall (macro): 0.5368
  ARI: 0.4291
  NMI: 0.4618
--------------------------------------------------


c:\Users\ADMIN\anaconda3\envs\tfm\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Saved predictions to: Mosta_ML_labels\sgd_labels.csv
SGD metrics:
  Accuracy: 0.7744
  F1 (macro): 0.6295
  Precision (macro): 0.6633
  Recall (macro): 0.6117
  ARI: 0.5228
  NMI: 0.5433
--------------------------------------------------


c:\Users\ADMIN\anaconda3\envs\tfm\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Saved predictions to: Mosta_ML_labels\decisiontree_labels.csv
DecisionTree metrics:
  Accuracy: 0.5118
  F1 (macro): 0.4119
  Precision (macro): 0.4287
  Recall (macro): 0.4005
  ARI: 0.1862
  NMI: 0.2342
--------------------------------------------------


c:\Users\ADMIN\anaconda3\envs\tfm\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Saved predictions to: Mosta_ML_labels\naivebayes_labels.csv
NaiveBayes metrics:
  Accuracy: 0.4556
  F1 (macro): 0.3364
  Precision (macro): 0.3563
  Recall (macro): 0.3560
  ARI: 0.1909
  NMI: 0.1909
--------------------------------------------------


c:\Users\ADMIN\anaconda3\envs\tfm\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
